In [1]:
import sys, os
from typing import Optional
import numpy as np
import pandas as pd
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/yuyuezhu/Desktop/FRE-GT-9743-Assignment-5
Fixed Income Library is loaded.


# Instructions for HW5:

## PCA implementation

### 1. Write PCA algorithm

Please implement a PCA algorithm without using existing PCA api, which means you're only allowed to use Numpy package.

In [21]:
def my_pca(X, num_components):
    # Data centering/standardization
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X_centered = (X - X_mean) / X_std
    
    # Covariance matrix computation
    cov_matrix = np.cov(X_centered, rowvar=False)
    
    # Eigenvalue/eigenvector calculation
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    
    # Sort eigenvalues and eigenvectors in descending order
    idx = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[idx]
    sorted_eigenvectors = eigenvectors[:, idx]
    
    # Projection
    X_pca = np.dot(X_centered, sorted_eigenvectors[:, :num_components])
    
    # RETURN ALL THREE HERE
    return X_pca, sorted_eigenvalues, sorted_eigenvectors


### 2. Apply PCA

Please download fixed-floating swap rate data from Bloomberg (Ticker: YCSW0490) covering at least six months, with the sample period ending on 2026-02-26.

Although you should download the data for 2026-02-27, exclude that date from the PCA analysis.

The swap tenors should range from 1Y to 30Y.

The dataset should be organized in the format shown below.

After preparing the dataset, apply Principal Component Analysis (PCA) as discussed in class.

![Chart](data_sample.png)

In [22]:
!pip install openpyxl

file_path = '/Users/yuyuezhu/Desktop/grid1_wbpgglft.xlsx'
df = pd.read_excel(file_path)

# Filter: Exclude 2026-02-27
df['Dates'] = pd.to_datetime(df['Dates'])
df = df[df['Dates'] != '2026-02-27']

# Column Selection: Tenors 1Y (12M) to 30Y 
# We explicitly list them to skip '18M' and 'Dates'
feature_columns = ['12M', '2Y', '3Y', '4Y', '5Y', '6Y', '7Y', '8Y', 
                   '9Y', '10Y', '12Y', '15Y', '20Y', '25Y', '30Y']
features = df[feature_columns].values

# Calculate the first 3 components
X_pca, vals, vecs = my_pca(features, 3)

explained_var = vals / np.sum(vals)
print(f"PC1 (Level): {explained_var[0]:.2%}")
print(f"PC2 (Slope): {explained_var[1]:.2%}")
print(f"PC3 (Curvature): {explained_var[2]:.2%}")


PC1 (Level): 86.87%
PC2 (Slope): 10.72%
PC3 (Curvature): 2.31%


### 3. Hedging with PCA

With your PCA results, please try to reconstruct the risk profile provided, and present your results.

In [23]:
# read risk_agg.csv
risk_agg = pd.read_csv(os.path.join(repo_root,"risk_agg.csv"))
print("Risk Aggregation Data:")
risk_agg.head()

Risk Aggregation Data:


,Unnamed: 0,1Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,12Y,15Y,20Y,25Y,30Y
0,AGG_RISK,2.982972,-1.857554,0.013279,0.071185,13.975129,-5.458591,-0.025772,-7.003277,0.043624,12.783767,-15.256859,-21.13873,-17.444482,0.0,0.0


In [60]:
# prepare the risk vector (ensure it matches your PCA feature columns)
risk_vector = risk_agg.drop(columns=['Unnamed: 0']).values.flatten()

# extract top 3 eigenvectors (vecs from your previous step)
top_3_vecs = vecs[:, :3]

# calculate Factor Sensitivities (Exposures)
factor_exposures = np.dot(risk_vector, top_3_vecs)

# reconstruct the risk profile back into tenor space
reconstructed_risk = np.dot(factor_exposures, top_3_vecs.T)

# presentation: Compare Original vs Reconstructed
comparison_df = pd.DataFrame({
    'Tenor': feature_columns,
    'Original_Risk': risk_vector,
    'Reconstructed_Risk': reconstructed_risk,
    'Residual (Error)': risk_vector - reconstructed_risk
})

print(comparison_df)

   Tenor  Original_Risk  Reconstructed_Risk  Residual (Error)
0    12M       2.982972           -0.919295          3.902267
1     2Y      -1.857554            2.032390         -3.889944
2     3Y       0.013279            2.287415         -2.274136
3     4Y       0.071185            1.546450         -1.475264
4     5Y      13.975129            0.615697         13.359432
5     6Y      -5.458591           -0.391850         -5.066741
6     7Y      -0.025772           -1.384614          1.358842
7     8Y      -7.003277           -2.283132         -4.720146
8     9Y       0.043624           -3.109370          3.152994
9    10Y      12.783767           -3.876142         16.659909
10   12Y     -15.256859           -5.176249        -10.080610
11   15Y     -21.138730           -6.517306        -14.621423
12   20Y     -17.444482           -7.709555         -9.734927
13   25Y       0.000000           -8.397391          8.397391
14   30Y       0.000000           -8.818010          8.818010


Using the provided hedging instruments, solve for the optimal portfolio weights that hedge the aggregated DV01 risk profile. 

Compare the eigenspace approach against direct least squares, and argue mathematically why the former is preferred.

In [45]:
hedging = pd.read_csv(os.path.join(repo_root,"hedging_instruments.csv"))
print("Hedging Instruments Data:")
hedging.head()

Hedging Instruments Data:


,Unnamed: 0,1Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,12Y,15Y,20Y,25Y,30Y
0,1Y1Y,0.990383,-1.905780,-0.040437,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
1,1Y5Y,0.986415,0.006578,0.000573,0.001714,0.049129,-5.468511,-0.035436,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
2,5Y5Y,0.002970,0.005982,0.009149,0.012336,4.626952,-0.016352,-0.019360,-0.022309,-0.025584,-8.501197,0.000000,0.000000,0.000000,0.0,0.0
3,10Y10Y,0.003699,0.007465,0.011399,0.015366,0.019642,0.023980,0.028543,0.032885,0.037736,8.499983,-0.093768,-0.198309,-14.335121,0.0,0.0


In [ ]:
# prepare Data Containers
# risk profiles of the 4 hedging instruments
H = hedging.drop(columns=['Unnamed: 0']).values 

# your portfolio aggregate risk vector
R = risk_agg.drop(columns=['Unnamed: 0']).values.flatten()

# top 3 eigenvectors (Level, Slope, Curvature)
V_k = vecs[:, :3]

# project everything into the 3D PCA space
# H_pca will be
H_pca = np.dot(H, V_k) 
# R_pca will be
R_pca = np.dot(R, V_k)

# solve for weights: H_pca.T * weights = R_pca
weights_pca, _, _, _ = np.linalg.lstsq(H_pca.T, R_pca, rcond=None)

# solve for weights: H.T * weights = R
weights_ls, _, _, _ = np.linalg.lstsq(H.T, R, rcond=None)

hedging_names = hedging['Unnamed: 0'].values
results = pd.DataFrame({
    'Instrument': hedging_names,
    'PCA_Hedge_Weights': weights_pca,
    'Direct_LS_Weights': weights_ls
})

print("Optimal Hedging Weights Comparison:")
print(results)

# check how much Level/Slope/Curvature risk remains after PCA hedge
remaining_pca_risk = R_pca - np.dot(weights_pca, H_pca)
print("\nRemaining Factor Risk (PCA approach):")
print(f"Level: {remaining_pca_risk[0]:.4f}, Slope: {remaining_pca_risk[1]:.4f}, Curvature: {remaining_pca_risk[2]:.4f}")


Optimal Hedging Weights Comparison:
  Instrument  PCA_Hedge_Weights  Direct_LS_Weights
0       1Y1Y           1.940106           1.193592
1       1Y5Y           1.974920           1.045629
2       5Y5Y           2.307850           0.674989
3     10Y10Y           3.102386           1.487148

Remaining Factor Risk (PCA approach):
Level: -0.0000, Slope: -0.0000, Curvature: 0.0000


I believe the Eigenspace approach is better than Direct Least Squares because it focuses on the big market moves instead of small, random errors. In the swap market, different tenors are highly correlated, so rates for 9-year and 10-year swaps usually move together. Direct Least Squares tries to match the risk at every single point on the curve exactly. This often makes the math unstable, which leads to a portfolio with huge long and short positions that just cancel each other out. These positions are expensive to trade and way too sensitive to noise in the data.

By projecting the risk onto the first three principal components, the Eigenspace approach filters the data to focus only on Level, Slope, and Curvature. This makes the problem much simpler and the hedging results more reliable. This method captures about 99% of how the market actually moves. Because of this, the hedge weights are much more stable, easier to manage in a real portfolio, and better at protecting against broad shifts in interest rates.